# 01 — LangChain Basics

This notebook covers LangChain Expression Language (LCEL), chains, prompt templates, memory, and runnable composition.

**Kernel:** `ai-engineering`

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Verify API key is set
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"

In [ ]:
from langchain_openai import ChatChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage

model = ChatChatOpenAI(model="gpt-4o-mini", temperature=0.7)

## 1. Simple Chain with LCEL

LCEL uses the `|` pipe operator to compose runnables. Each runnable accepts input and returns output.

```
PromptTemplate | ChatModel | OutputParser
```

The input flows left to right through each component.

In [ ]:
# Simple chain: prompt → model → parser
chain = (
    ChatPromptTemplate.from_template("Explain {topic} to a 10-year-old in 2 sentences.") 
    | model 
    | StrOutputParser()
)

result = chain.invoke({"topic": "black holes"})
print(result)

## 2. Multi-Step Chain

Chain the output of one step into the input of the next. Here we generate a poem, then analyze it.

In [ ]:
# Step 1: Generate a poem
generate_poem = (
    ChatPromptTemplate.from_template("Write a short poem about {topic}. Keep it to 4 lines.") 
    | model 
    | StrOutputParser()
)

# Step 2: Analyze the poem
analyze_poem = (
    ChatPromptTemplate.from_template(
        "Analyze this poem. Identify the tone, main imagery, and emotional effect:\n\n{poem}"
    ) 
    | model 
    | StrOutputParser()
)

# Compose: generate → analyze
full_chain = {"poem": generate_poem} | analyze_poem

result = full_chain.invoke({"topic": "the ocean"})
print(result)

## 3. RunnableParallel

Run multiple chains concurrently on the same input. Results come back as a dict.

In [ ]:
parallel_chain = RunnableParallel(
    pros=(
        ChatPromptTemplate.from_template("List 3 pros of {topic}.") 
        | model 
        | StrOutputParser()
    ),
    cons=(
        ChatPromptTemplate.from_template("List 3 cons of {topic}.") 
        | model 
        | StrOutputParser()
    ),
    summary=(
        ChatPromptTemplate.from_template("In one sentence, summarize the debate around {topic}.") 
        | model 
        | StrOutputParser()
    )
)

result = parallel_chain.invoke({"topic": "remote work"})
for key, value in result.items():
    print(f"=== {key.upper()} ===")
    print(value)
    print()

## 4. RunnablePassthrough

Pass the original input through unchanged alongside processed output. Useful when you need to preserve the original data.

In [ ]:
chain_with_passthrough = RunnableParallel(
    original=RunnablePassthrough(),
    rewriter=(
        ChatPromptTemplate.from_template(
            "Rewrite this text in a formal professional tone:\n\n{text}"
        ) 
        | model 
        | StrOutputParser()
    )
)

result = chain_with_passthrough.invoke({"text": "hey whats up, wanna grab lunch later?"})
print("Original:", result["original"]["text"])
print("\nRewritten:", result["rewriter"])

## 5. Prompt Templates

Templates allow reusable, parameterized prompts. `ChatPromptTemplate` supports system + human messages.

In [ ]:
# System + Human message template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Respond in character."),
    ("human", "{question}")
])

chain = prompt | model | StrOutputParser()

result = chain.invoke({
    "role": "pirate",
    "question": "What do you think about the cloud?"
})
print(result)

In [ ]:
# Few-shot prompt template
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "excited", "output": "🚀 Enthusiastic anticipation"},
    {"input": "sad", "output": "🌧️ Melancholic reflection"},
    {"input": "angry", "output": "🔥 Fiery indignation"},
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "Convert emotions to poetic descriptions."),
    few_shot,
    ("human", "{input}")
])

chain = final_prompt | model | StrOutputParser()

result = chain.invoke({"input": "nervous"})
print(result)

## 6. Memory — Conversation History

For multi-turn conversations, maintain a list of messages using `MessagesPlaceholder`.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful math tutor. Be concise."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = prompt | model | StrOutputParser()

# Conversation simulation
history = []

# Turn 1
user_input = "What is a derivative?"
response = chain.invoke({"input": user_input, "history": history})
print(f"User: {user_input}")
print(f"AI: {response}\n")

# Update history
history.append(HumanMessage(content=user_input))
history.append(AIMessage(content=response))

# Turn 2 — model should remember the context
user_input = "Can you give me an example?"
response = chain.invoke({"input": user_input, "history": history})
print(f"User: {user_input}")
print(f"AI: {response}\n")

history.append(HumanMessage(content=user_input))
history.append(AIMessage(content=response))

# Turn 3
user_input = "How does this relate to what we discussed?"
response = chain.invoke({"input": user_input, "history": history})
print(f"User: {user_input}")
print(f"AI: {response}")

## 7. Advanced Composition

Combine parallel chains, passthrough, and sequential chains for complex pipelines.

In [ ]:
# Research pipeline: gather info + generate questions + synthesize

gather_info = (
    ChatPromptTemplate.from_template("Explain {topic} in 3 bullet points.") 
    | model 
    | StrOutputParser()
)

generate_questions = (
    ChatPromptTemplate.from_template(
        "Based on this information:\n\n{info}\n\nGenerate 2 follow-up questions."
    ) 
    | model 
    | StrOutputParser()
)

# Compose: gather info → parallel (info + questions) → final synthesis
pipeline = (
    {"info": gather_info}
    | RunnableParallel(
        info=RunnablePassthrough(),
        questions=lambda x: generate_questions.invoke(x)
    )
    | (lambda x: (
        ChatPromptTemplate.from_template(
            "Given this info:\n{info}\n\nAnd these questions:\n{questions}\n\n"
            "Write a brief study guide."
        ) | model | StrOutputParser()
    ).invoke(x))
)

result = pipeline.invoke({"topic": "recursion in programming"})
print(result)

## Key Takeaways

| Concept | Pattern | Use Case |
|---------|---------|----------|
| Simple chain | `prompt \| model \| parser` | Single-step tasks |
| Multi-step | `{"key": chain} \| chain` | Chained pipelines |
| RunnableParallel | `RunnableParallel(a=..., b=...)` | Concurrent execution |
| RunnablePassthrough | `RunnablePassthrough()` | Preserve original input |
| Memory | `MessagesPlaceholder` | Multi-turn conversation |
| Few-shot | `FewShotChatMessagePromptTemplate` | Example-based prompting |

Next: [02-tool-calling.ipynb](02-tool-calling.ipynb) — connecting external tools to LLMs.